In [1]:
from ERA_Distribution_Classes_Python.Classes.ERADist import ERADist
from ERA_Distribution_Classes_Python.Classes.ERANataf import ERANataf
from ERA_Distribution_Classes_Python.Classes.FORM_HLRF import FORM_HLRF
from ERA_Distribution_Classes_Python.Classes.FORM_fmincon import FORM_fmincon
from ERA_Distribution_Classes_Python.Classes.SuS import SuS

In [2]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt 
from structure import Structure
from solver_2nd import Solver2ndOrder
from syst_4_model_functions import t_S_nonlinear

## Material Properties

In [3]:
# NEW modified IPE 120 -> eta = 100% for Th.I.O.
E = 210e6 # kN/m2 
A = 1.321e-3 # m2
I = 0.2740555e-5 # m4
h = 0.12  # m
z = h/2  # m
sigma_yield = 35.5 # kN/cm2 
alpha = 1.14
M_yield = I/z * sigma_yield * 100e2 * 1.14 # kNm

In [4]:
def t_R(M_k, I=I, z=z, alpha=alpha):
    """
    Takes in the Steel Bending Strength M_k (random variable) in kN/cm2
    I in m^4
    z in m
    alpha is the plastic ratio 

    Returns the characteristic Bending Moment Resistance M_c_Rk for the given system in kNm
    """
    return I/z * M_k * 100e2 * alpha

## System Definition

In [10]:
# Design Opt 1
print(t_S_nonlinear(l_1= 1.1*1.5, l_2= 0.65*1.5) * 1.0 / t_R(35.5))

# Design Opt 2
e_d = max(1.5 * t_S_nonlinear(l_1= 1.1, l_2= 0.65), 1.5 * t_S_nonlinear(l_1= 1.1, l_2= 0.65))
print(e_d * 1.0 / t_R(35.5))

1.1452427630413018
1.0964527164811924


In [11]:
# Vectorized Version of the Structural response function (Better for array handling later)
t_S_vectorized = np.vectorize(t_S_nonlinear, otypes=[float])

## Characteristic Values (for calibrating the probabilistic loads)

In [ ]:
s_k = 1.1 # snow load on ground kN/m2
q_b = 0.65 # wind pressure kN/m2
w_k = q_b * 0.8 # wind load kN/m2 with c_pe,10 = 0.8 (Area D)
m_k = 35.5 # kN/cm2 (steel yield resistance)

In [13]:
# Quick Check: should give 21.17 kNm 
print(t_S_nonlinear(l_1=s_k * 1.5, l_2=q_b * 1.5))

21.169862264247584


## Distributions

In [14]:
# Snow time-invariant part
mu_Theta_1 = 0.81
cov_Theta_1 = 0.26
sig_Theta_1 = mu_Theta_1 * cov_Theta_1
Theta_L1 = ERADist('lognormal','MOM',[mu_Theta_1, sig_Theta_1])

# Snow load on ground
mu_L1 = 1.0
cov_L1 = 0.2
sig_L1 = mu_L1 * cov_L1
L1 = ERADist('gumbel','MOM',[mu_L1,sig_L1])

In [15]:
percentile_L1 = L1.icdf(0.98)
print(f"Snow 98% Percentile: {percentile_L1}")

Snow 98% Percentile: 1.5184551765313796


In [16]:
# Wind time-invariant part
mu_Theta_2 = 0.97
cov_Theta_2 = 0.26
sig_Theta_2 = mu_Theta_2 * cov_Theta_2
Theta_L2 = ERADist('lognormal','MOM',[mu_Theta_2, sig_Theta_2])

# Wind velocity pressure
mu_L2 = 1.0 
cov_L2 = 0.14
sig_L2 = mu_L2 * cov_L2
L2 = ERADist('gumbel','MOM',[mu_L2, sig_L2])

In [17]:
percentile_L2 = L2.icdf(0.98)
print(f"Wind 98% Percentile: {percentile_L2}")

Wind 98% Percentile: 1.3629186235719657


In [18]:
# Structural Response Model Uncertainty (from JCSS Probabilistic Model Code, Part 3, Table 3.9.1)
mu_Theta_S = 1.0
cov_Theta_S = 0.1
sig_Theta_S = mu_Theta_S * cov_Theta_S
Theta_S = ERADist('lognormal','MOM',[mu_Theta_S, sig_Theta_S])  # Distribution for Moments in frames

In [19]:
# Steel bending model uncertainty
mu_Theta_M = 1.15
cov_Theta_M = 0.05
sig_Theta_M = mu_Theta_M * cov_Theta_M
Theta_M = ERADist('lognormal','MOM',[mu_Theta_M, sig_Theta_M])

# Steel yielding strength
mu_M = 1.0
cov_M = 0.05
sig_M = mu_M * cov_M
M = ERADist('lognormal','MOM',[mu_M, sig_M])

In [20]:
percentile_M = M.icdf(0.05)
print(f"Steel 5% Percentile: {percentile_M}")

Steel 5% Percentile: 0.9199464756612658


## Shifting / Scaling Distributions

In [21]:
# Snow Load on Ground, shifted to characteristic value
snow_shift = s_k / percentile_L1 # ratio of target to current percentile, by which mean and std get multiplied

mu_L1_shifted = mu_L1 * snow_shift
sig_L1_shifted = sig_L1 * snow_shift
L1_shifted = ERADist('gumbel','MOM',[mu_L1_shifted, sig_L1_shifted])

print(f"""Snow Load on Ground gets shifted by {snow_shift}""")
print(f"""Old mean: {mu_L1}; New mean: {mu_L1_shifted}""")
print(f"""Old std: {sig_L1}; New std: {sig_L1_shifted}""")
print(f"""Old 98th percentile: {L1.icdf(.98)}; New 98th percentile: {L1_shifted.icdf(.98)}""")
print(f"""Old COV: {L1.std()/L1.mean()}; New COV: {L1_shifted.std()/L1_shifted.mean()}""")

Snow Load on Ground gets shifted by 0.7244204616646898
Old mean: 1.0; New mean: 0.7244204616646898
Old std: 0.2; New std: 0.14488409233293795
Old 98th percentile: 1.5184551765313796; New 98th percentile: 1.0999999999999999
Old COV: 0.19999999999999998; New COV: 0.19999999999999996


In [22]:
# Wind velocity pressure, shifted to characteristic value
wind_shift = q_b / percentile_L2

mu_L2_shifted = mu_L2 * wind_shift
sig_L2_shifted = sig_L2 * wind_shift
L2_shifted = ERADist('gumbel','MOM',[mu_L2_shifted, sig_L2_shifted])

print(f"""Wind velocity pressure gets shifted by {wind_shift}""")
print(f"""Old mean: {mu_L2}; New mean: {mu_L2_shifted}""")
print(f"""Old std: {sig_L2}; New std: {sig_L2_shifted}""")
print(f"""Old 98th percentile: {L2.icdf(.98)}; New 98th percentile: {L2_shifted.icdf(.98)}""")
print(f"""Old COV: {L2.std()/L2.mean()}; New COV: {L2_shifted.std()/L2_shifted.mean()}""")

Wind velocity pressure gets shifted by 0.4769176888173018
Old mean: 1.0; New mean: 0.4769176888173018
Old std: 0.14; New std: 0.06676847643442226
Old 98th percentile: 1.3629186235719657; New 98th percentile: 0.65
Old COV: 0.14; New COV: 0.13999999999999999


In [23]:
# Steel bending resistance, shifted to characteristic value
steel_shift = m_k / percentile_M

mu_M_shifted = mu_M * steel_shift
sig_M_shifted = sig_M * steel_shift
M_shifted = ERADist('lognormal','MOM',[mu_M_shifted, sig_M_shifted])

print(f"""Steel bending resistance gets shifted by {steel_shift}""")
print(f"""Old mean: {mu_M}; New mean: {mu_M_shifted}""")
print(f"""Old std: {sig_M}; New std: {sig_M_shifted}""")
print(f"""Old 5th percentile: {M.icdf(0.05)}; New 5th percentile: {M_shifted.icdf(0.05)}""")
print(f"""Old COV: {M.std()/M.mean()}; New COV: {M_shifted.std()/M_shifted.mean()}""")

Steel bending resistance gets shifted by 38.58920158858403
Old mean: 1.0; New mean: 38.58920158858403
Old std: 0.05; New std: 1.9294600794292016
Old 5th percentile: 0.9199464756612658; New 5th percentile: 35.5
Old COV: 0.04999999999999947; New COV: 0.04999999999999946


### Distribution Plots

In [24]:
x_plotting = np.linspace(0, 2.0, 200) # for PDF plotting only
x_resistance_plotting = np.linspace(30, 50, 200) # for PDF plotting only

In [25]:
# # FIGURE 1: LOADS Side by side
# # ================================================================================
# fig1, axes = plt.subplots(1, 2, figsize=(16, 5))

# # --- Left Plot: SNOW ---
# axes[0].plot(x_plotting, Theta_L1.pdf(x_plotting), label=fr'$\Theta_1$ (Lognormal, $\mu={Theta_L1.mean():.2f}, COV={Theta_L1.std()/Theta_L1.mean():.2f}$)')
# axes[0].plot(x_plotting, L1.pdf(x_plotting), label=fr'$Q_1$ (Gumbel, $\mu={L1.mean():.2f}, COV={L1.std()/L1.mean():.2f}$)')
# axes[0].plot(x_plotting, L1_shifted.pdf(x_plotting), label=fr'$Q_1,shifted$ (Gumbel, $\mu={L1_shifted.mean():.2f}, COV={L1_shifted.std()/L1_shifted.mean():.2f}$)', linestyle= "--", color = "orange")
# axes[0].set_title('Snow Load Components')
# axes[0].set_xlabel('$x$')
# axes[0].set_ylabel('$f(x)$')
# axes[0].grid(True, linestyle='--', alpha=0.6)
# axes[0].legend(fontsize="medium")
# axes[0].set_ylim(top=axes[0].get_ylim()[1] * 1.25)

# # --- Right Plot: WIND ---
# axes[1].plot(x_plotting, Theta_L2.pdf(x_plotting), label=fr'$\Theta_2$ (Lognormal, $\mu={Theta_L2.mean():.2f}, COV={Theta_L2.std()/Theta_L2.mean():.2f}$)')
# axes[1].plot(x_plotting, L2.pdf(x_plotting), label=fr'$Q_2$ (Gumbel, $\mu={L2.mean():.2f}, COV={L2.std()/L2.mean():.2f}$)')
# axes[1].plot(x_plotting, L2_shifted.pdf(x_plotting), label=fr'$Q_2,shifted$ (Gumbel, $\mu={L2_shifted.mean():.2f}, COV={L2_shifted.std()/L2_shifted.mean():.2f}$)', linestyle= "--", color = "orange")
# axes[1].set_title('Wind Load Components')
# axes[1].set_xlabel('$x$')
# axes[1].set_ylabel('$f(x)$')
# axes[1].grid(True, linestyle='--', alpha=0.6)
# axes[1].legend(fontsize="medium")
# axes[1].set_ylim(top=axes[1].get_ylim()[1] * 1.25)

# fig1.tight_layout()
# plt.show()


In [26]:
# # FIGURE 2: Resistance
# # ================================================================================
# fig2, axes = plt.subplots(1, 2, figsize=(16, 5))

# # --- Left Plot: Resistance Original ---
# axes[0].plot(x_plotting, Theta_M.pdf(x_plotting), label=fr'$\Theta_M$ (Lognormal, $\mu={Theta_M.mean():.2f}, COV={Theta_M.std()/Theta_M.mean():.2f}$)', color='blue')
# axes[0].plot(x_plotting, M.pdf(x_plotting), label=fr'$M$ (Lognormal, $\mu={M.mean():.2f}, COV={M.std()/M.mean():.2f}$)', color='orange')
# axes[0].set_title('Steel Yielding Strength (Resistance)')
# axes[0].set_xlabel('$x$')
# axes[0].set_ylabel('$f(x)$')
# axes[0].grid(True, linestyle='--', alpha=0.6)
# axes[0].legend(fontsize="medium")
# axes[0].set_ylim(top=axes[0].get_ylim()[1] * 1.25)

# # --- Right Plot: Resistance Shifted (without Model uncertainty) --
# axes[1].plot(x_resistance_plotting, M_shifted.pdf(x_resistance_plotting), label=fr'$M,shifted$ (Lognormal, $\mu={M_shifted.mean():.2f}, COV={M_shifted.std()/M_shifted.mean():.2f}$)', color='orange', linestyle= "--")
# axes[1].set_title('Steel Yielding Strength (Resistance)')
# axes[1].set_xlabel('$x$')
# axes[1].set_ylabel('$f(x)$')
# axes[1].grid(True, linestyle='--', alpha=0.6)
# axes[1].legend(fontsize="medium")
# axes[1].set_ylim(top=axes[1].get_ylim()[1] * 1.25)

# fig1.tight_layout()
# plt.show()

## Partial Safety Factors

In [27]:
gamma_M = 1.0  # Resistance
gamma_F1 = 1.5   # Snow Load
gamma_F2 = 1.5   # Wind Load
#psi_0 = 1.0 # 0.6     # Windload

In [28]:
# Array of marginal distributions
marginal_dist = [Theta_M, M_shifted, Theta_L1, L1_shifted, Theta_L2, L2_shifted, Theta_S]

# Correlation matrix (no correlation yet)
dimensions = len(marginal_dist)
R_xx = np.eye(dimensions)

# Construction of the Nataf Distribution
nataf = ERANataf(M=marginal_dist, Correlation=R_xx)

## Subset Simulation

### Design Option 1

In [29]:
# deterministic design action effect
e_d_opt1 = t_S_nonlinear(l_1=gamma_F1 * s_k, l_2=gamma_F2 * q_b) # kNm

In [30]:
def g_opt_1_sus(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k) # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt1 / r_k) * x[:,0] * t_R(M_k=x[:,1])
    action_side = x[:,6] * t_S_vectorized(l_1=(x[:,2] * x[:,3]), l_2=(x[:,4] * x[:,5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [31]:
# # %% Samples Return
# samples_return = 1
# # %% subset simulation
# N  = 10000        # Total number of samples for each level
# p0 = 0.1         # Probability of each subset, chosen adaptively

# print('\n\nSUBSET SIMULATION: ')
# [Pf_SuS_1, delta_SuS, b, Pf, b_sus, pf_sus, samplesU, samplesX, fs_iid] = SuS(N, p0, g_opt_1_sus, nataf, samples_return)

In [32]:
# print("Subset Simulation for Design Option 1")
# print(f"P(F) = {Pf_SuS_1}")
# X = sp.stats.Normal()
# beta = - X.icdf(Pf_SuS_1)
# print(f"beta = {beta}")
# print(samplesX)

---

### Design Option 2

In [33]:
# deterministic design action effect
argument_1 = gamma_F1 * t_S_nonlinear(l_1= s_k, l_2= (gamma_F2 / gamma_F1) * q_b)
argument_2 = gamma_F2 * t_S_nonlinear(l_1= (gamma_F1 / gamma_F2) * s_k, l_2= q_b)
e_d_opt2 = max(argument_1, argument_2) # kNm

In [34]:
def g_opt_2_sus(x):
    """
    LSF for Design Option 2
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt2 / r_k) * x[:,0] * t_R(M_k=x[:,1])
    action_side = x[:,6] * t_S_vectorized(l_1=(x[:,2] * x[:,3]), l_2=(x[:,4] * x[:,5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [35]:
# # %% Samples Return
# samples_return = 1
# # %% subset simulation
# N  = 10000        # Total number of samples for each level
# p0 = 0.1         # Probability of each subset, chosen adaptively

# print('\n\nSUBSET SIMULATION: ')
# [Pf_SuS_2, delta_SuS, b, Pf, b_sus, pf_sus, samplesU, samplesX, fs_iid] = SuS(N, p0, g_opt_2_sus, nataf, samples_return)

In [36]:
# print("Subset Simulation for Design Option 2")
# print(f"P(F) = {Pf_SuS_2}")

# X = sp.stats.Normal()
# beta = - X.icdf(Pf_SuS_2)
# print(f"beta = {beta}")
# # print(samplesX)

## FORM Analysis

### Design Option 1

In [37]:
def g_opt_1_FORM(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k) # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt1 / r_k) * x[0] * t_R(M_k=x[1])
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [38]:
# Perform FORM with HLRF
# [u_star, x_star, beta, Pf, S_F1, S_F1_T] = FORM_HLRF(g=g_opt_1, dg=[], distr=nataf, sensitivity_analysis=0, u0=0)

# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_1_FORM, dg=[], distr=nataf, u0=0)


*scipy.optimize.minimize() with  SLSQP  Method

  24  iterations... Reliability index =  3.7943299834896016  --- Failure probability =  7.402134064237337e-05 




In [39]:
print(f"u_star = {u_star}")
print(f"x_star = {x_star}")
print(f"(alpha_2)^2 = {(u_star/beta)**2}")
print(f"beta = {beta}")
print(f"P(F) = {Pf}")
print(f"g(X*) = {g_opt_1_FORM(x_star)}")

u_star = [-0.59643061 -0.59643083  0.26409327  0.19133514  2.75410255  2.13905308
  1.19100693]
x_star = [ 1.11483965 37.40936661  0.838716    0.72636531  1.89880223  0.6610196
  1.12056092]
(alpha_2)^2 = [0.02470869 0.0247087  0.00484445 0.00254284 0.52685368 0.31781393
 0.09852771]
beta = 3.7943299834896016
P(F) = 7.402134064237337e-05
g(X*) = 3.4214021837897235e-08


### Design Option 2

In [40]:
def g_opt_2_FORM(x):
    """
    LSF for Design Option 2
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * e_d_opt2 / r_k) * x[0] * t_R(M_k=x[1])
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [41]:
# Perform FORM with HLRF
# [u_star, x_star, beta, Pf, S_F1, S_F1_T] = FORM_HLRF(g=g_opt_2, dg=[], distr=nataf, sensitivity_analysis=0, u0=1)

# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_2_FORM, dg=[], distr=nataf)

KeyboardInterrupt: 

In [ ]:
print(f"u_star = {u_star}")
print(f"x_star = {x_star}")
print(f"(alpha_2)^2 = {(u_star/beta)**2}")
print(f"beta = {beta}")
print(f"P(F) = {Pf}")
print(f"g(X*) = {g_opt_2_FORM(x_star)}")

u_star = [-0.57548846 -0.5754885   0.25484562  0.1886141   2.66953752  2.04143005
  1.14825919]
x_star = [ 1.11600689 37.44853463  0.83673463  0.72598664  1.85817521  0.64843497
  1.11579286]
(alpha_2)^2 = [0.02476116 0.02476116 0.0048557  0.00265978 0.5328068  0.31157795
 0.09857745]
beta = 3.6572205210668414
P(F) = 0.00012748250858593953
g(X*) = 1.0898486380028771e-07


## Optimization of additional PSF $\gamma_{new}$

Remark: due to long simulation time with SuS, only FORM is used here

In [ ]:
from scipy.optimize import minimize
from scipy.optimize import brentq

In [ ]:
beta_TRG = 5.230751 # TH1 FORM solution, copied from 4_reliability_analysis_lin.ipynb

### Design Option (1) 

In [ ]:
def f(gamma_new):
    def g_opt_1(x):
        """
        LSF for Design Option 1
        Input variables: 
        x[0]: Theta_M = Resistance Model Uncertainty
        x[1]: M = Steel yield strength
        x[2]: Theta_L1 = Snow Load Model Uncertainty
        x[3]: L1 = Snow Load on Ground
        x[4]: Theta_L2 = Wind Load Model Uncertainty
        x[5]: L2 = Wind velocity pressure
        x[6]: Theta_S = Structural Response Model Uncertainty
        """
        
        # deterministic characteristic resistance 
        r_k = t_R(M_k=m_k) # kNm
        
        # assembly of the LSF
        resistance_side = (gamma_M * gamma_new *e_d_opt1 / r_k) * x[0] * t_R(M_k=x[1])
        action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
        
        # print(f"resistance = {resistance_side}")
        # print(f"action = {action_side}")
        
        return resistance_side - action_side

    [u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_1, dg=[], distr=nataf)
    return beta - beta_TRG

In [ ]:
# this is for narrowing down the lower and upper bound of where brentq should search in the input space
# for g in [0.8, 0.85 ,0.9]:
#     print(g, f(g))


*scipy.optimize.minimize() with  SLSQP  Method

  21  iterations... Reliability index =  5.131426675267906  --- Failure probability =  1.4377712152931287e-07 


0.8 -0.09932432473209385


/var/folders/lj/y31_mc2n4cj8z4jb_s7xdnqr0000gn/T/ipykernel_97946/2800514093.py:25: RuntimeWarning: invalid value encountered in scalar subtract
  return resistance_side - action_side



*scipy.optimize.minimize() with  SLSQP  Method

  29  iterations... Reliability index =  5.3193919350507075  --- Failure probability =  5.205731592316662e-08 


0.85 0.0886409350507078

*scipy.optimize.minimize() with  SLSQP  Method

  23  iterations... Reliability index =  5.4968358725292585  --- Failure probability =  1.9333313251443126e-08 


0.9 0.26608487252925883


Observation: root lies between $x = 0.8$ and $x = 0.85$


In [ ]:
# Finding the Root (Optimization Problem)
gamma_new_1 = brentq(f, 0.8, 0.85)


*scipy.optimize.minimize() with  SLSQP  Method

  21  iterations... Reliability index =  5.131426675267906  --- Failure probability =  1.4377712152931287e-07 




/var/folders/lj/y31_mc2n4cj8z4jb_s7xdnqr0000gn/T/ipykernel_97946/2800514093.py:25: RuntimeWarning: invalid value encountered in scalar subtract
  return resistance_side - action_side



*scipy.optimize.minimize() with  SLSQP  Method

  29  iterations... Reliability index =  5.3193919350507075  --- Failure probability =  5.205731592316662e-08 



*scipy.optimize.minimize() with  SLSQP  Method

  28  iterations... Reliability index =  5.232139867870172  --- Failure probability =  8.377944188090753e-08 



*scipy.optimize.minimize() with  SLSQP  Method

  17  iterations... Reliability index =  5.230783275239267  --- Failure probability =  8.439665253099931e-08 



*scipy.optimize.minimize() with  SLSQP  Method

  23  iterations... Reliability index =  5.230728842513118  --- Failure probability =  8.442150937459604e-08 



*scipy.optimize.minimize() with  SLSQP  Method

  19  iterations... Reliability index =  5.230732518843822  --- Failure probability =  8.44198303459132e-08 



*scipy.optimize.minimize() with  SLSQP  Method

  25  iterations... Reliability index =  5.230760883530352  --- Failure probability =  8.440687690527645e-08 



*scipy.optimize.minimize() with  

In [ ]:
print(f"gamma_new = {gamma_new_1}")

gamma_new = 0.8260468918830731


### Verification of $\gamma_{new}$ for Design Option (1)

In [ ]:
def g_opt_1_verification(x):
    """
    LSF for Design Option 1
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k) # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * gamma_new_1 * e_d_opt1 / r_k) * x[0] * t_R(M_k=x[1])
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [ ]:
# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_1_verification, dg=[], distr=nataf, u0=0)


*scipy.optimize.minimize() with  SLSQP  Method

  24  iterations... Reliability index =  5.230736782135288  --- Failure probability =  8.44178832849455e-08 




In [ ]:
print(f"Target Reliability Index: {beta_TRG}")
print(f"Optimized Reliability Index: {beta}")
print(f"Difference: {beta - beta_TRG}")

Target Reliability Index: 5.230751
Optimized Reliability Index: 5.230736782135288
Difference: -1.4217864711341122e-05


### Design Option (2) 

In [ ]:
def f(gamma_new):
    def g_opt_2(x):
        """
        LSF for Design Option 2
        Input variables: 
        x[0]: Theta_M = Resistance Model Uncertainty
        x[1]: M = Steel yield strength
        x[2]: Theta_L1 = Snow Load Model Uncertainty
        x[3]: L1 = Snow Load on Ground
        x[4]: Theta_L2 = Wind Load Model Uncertainty
        x[5]: L2 = Wind velocity pressure
        x[6]: Theta_S = Structural Response Model Uncertainty
        """
        
        # deterministic characteristic resistance 
        r_k = t_R(M_k=m_k)  # kNm
        
        # assembly of the LSF
        resistance_side = (gamma_M * gamma_new * e_d_opt2 / r_k) * x[0] * t_R(M_k=x[1])
        action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
        
        # print(f"resistance = {resistance_side}")
        # print(f"action = {action_side}")
        
        return resistance_side - action_side

    [u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_2, dg=[], distr=nataf)
    return beta - beta_TRG

In [ ]:
# # this is for narrowing down the lower and upper bound of where brentq should search in the input space
# for g in [0.7, 0.8 ,0.9, 1.0]:
#     print(g, f(g))

Observation: root lies between $x = 0.8$ and $x = 0.9$. 
Narrowed down a little more to 0.85 and 0.87

In [ ]:
# Finding the Root (Optimization Problem)
# increased the tolerance, otherwise solver will test out infeasible high loads and crash
gamma_new_2 = brentq(f, a=0.85, b=0.87 ,xtol=1e-4, rtol=1e-6, maxiter=50)


*scipy.optimize.minimize() with  SLSQP  Method

  29  iterations... Reliability index =  5.184403808696989  --- Failure probability =  1.0835347705398408e-07 



*scipy.optimize.minimize() with  SLSQP  Method

  24  iterations... Reliability index =  5.256553391911387  --- Failure probability =  7.339008860797019e-08 



*scipy.optimize.minimize() with  SLSQP  Method

  24  iterations... Reliability index =  5.230951568990141  --- Failure probability =  8.431984551181257e-08 



*scipy.optimize.minimize() with  SLSQP  Method

  23  iterations... Reliability index =  5.230740253449278  --- Failure probability =  8.441629795508784e-08 




In [ ]:
print(f"gamma_new = {gamma_new_2}")

gamma_new = 0.8627917362693367


### Verification of $\gamma_{new}$ for Design Option (2)

In [ ]:
def g_opt_2_verification(x):
    """
    LSF for Design Option 2
    Input variables: 
    x[0]: Theta_M = Resistance Model Uncertainty
    x[1]: M = Steel yield strength
    x[2]: Theta_L1 = Snow Load Model Uncertainty
    x[3]: L1 = Snow Load on Ground
    x[4]: Theta_L2 = Wind Load Model Uncertainty
    x[5]: L2 = Wind velocity pressure
    x[6]: Theta_S = Structural Response Model Uncertainty
    """
    
    # deterministic characteristic resistance 
    r_k = t_R(M_k=m_k)  # kNm
    
    # assembly of the LSF
    resistance_side = (gamma_M * gamma_new_2 * e_d_opt2 / r_k) * x[0] * t_R(M_k=x[1])
    action_side = x[6] * t_S_vectorized(l_1=(x[2] * x[3]), l_2=(x[4] * x[5]))
    
    # print(f"resistance = {resistance_side}")
    # print(f"action = {action_side}")
    
    return resistance_side - action_side

In [ ]:
# Perform FORM with fmincom
[u_star, x_star, beta, alpha, Pf] = FORM_fmincon(g=g_opt_2_verification, dg=[], distr=nataf, u0=0)


*scipy.optimize.minimize() with  SLSQP  Method

  20  iterations... Reliability index =  5.230691015397001  --- Failure probability =  8.443878739244425e-08 




In [ ]:
print(f"Target Reliability Index: {beta_TRG}")
print(f"Optimized Reliability Index: {beta}")
print(f"Difference: {beta - beta_TRG}")

Target Reliability Index: 5.230751
Optimized Reliability Index: 5.230691015397001
Difference: -5.9984602998497394e-05
